<a href="https://colab.research.google.com/github/MelB18/EasyQuanten/blob/main/myMolM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%pip install --upgrade 'qiskit[visualization]>=2.5.0' qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 93.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 122.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 116.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [qiskit-aer]


In [7]:
%pip install rdkit py3Dmol

In [11]:
# --- IMPORTE: Wir laden alle benötigten Werkzeuge ---
import numpy as np # Bibliothek für mathematische Arrays und Zahlen
import pyscf # Klassisches Chemie-Programm zur Berechnung der Molekül-Basisdaten
import pyscf.cc # Modul für die klassische CCSD-Schätzung (Startwerte für den Schaltkreis)
import pyscf.fci # Modul für die exakte Vergleichslösung (zur Kontrolle am Ende)
import ffsim # Spezialbibliothek zum Bauen von Quantenchemie-Schaltkreisen
from qiskit import QuantumCircuit, QuantumRegister # Grundbausteine für Qiskit-Schaltkreise
from qiskit.primitives import StatevectorSampler # Lokaler Simulator, der einen Quantencomputer nachahmt



ModuleNotFoundError: No module named 'pyscf'

In [ ]:
from qiskit_addon_sqd.fermion import ( # IBMs neues SQD-Addon für die Auswertung
  SCIResult,
  diagonalize_fermionic_hamiltonian,
  solve_sci_batch,
  )

PySCF (Python-based Simulations of Chemistry Framework):

.build() ist eine Methode des gto.Mole Objekts, um die Molekülstruktur zu initialisieren.

Atome im 3D-Raum: (atom) aus welchen Elementen besteht das Atom und wo im Raum sitzen die Atomkerne.

Elektronen-Wolken: (basis)
Mit dem Basissatz legt man fest wie feinmaschig die Orbitale der Elektronen beschrieben werden.


größerer Basissatz => mehr Qubits => genauere Ergebnisse






# Wasserstoffmolekül

In [ ]:
# ==============================================================================
# SCHRITT 1: DAS MOLEKÜL DEFINIEREN & KLASSISCHE BASISDATEN BERECHNEN (PySCF)
# ==============================================================================

# Wir definieren das Wasserstoffmolekül (H2) mit zwei Atomen im Abstand von 0.74 Angström
mol = pyscf.gto.Mole()
mol.build(
  atom=[["H", (0.0, 0.0, 0.0)], ["H", (0.0, 0.0, 0.74)]], # Positionen der zwei H-Atome im Raum
  basis="6-31g" # Genauigkeitsstufe der Elektronen-Orbitale (ergibt 4 Orbitale = 8 Qubits)
)

# Lithiumhydrid

In [ ]:
# LiH mit sto-3g Basis benötigt 12 Qubits (6 Orbitale, 4 Elektronen)
# experimenteller Bindungsabstand von 1.595 Angström
mol = pyscf.gto.Mole()
mol.build(
  atom=[
      ["Li", (0.0, 0.0, 0.0)],
      ["H", (0.0, 0.0, 1.595)]
  ], # Positionen des H-Atomes im Raum
  basis="sto-3g" # Genauigkeitsstufe der Elektronen-Orbitale (ergibt 6 Orbitale = 12 Qubits)
)

NameError: name 'pyscf' is not defined

 # Wasser

In [ ]:
# Wassermolekül (H2O) in seiner gewinkelten Gleichgewichtsgeometrie

mol = pyscf.gto.Mole()
mol.build(
    atom=[
      ["O", (0.000, 0.000, 0.117)],
      ["H", (0.000, 0.757, -0.469)],
      ["H", (0.000, -0.757, -0.469)],
    ], # Positionen der drei H-Atome im Raum
    basis="sto-3g" # Genauigkeitsstufe der Elektronen-Orbitale (ergibt 7 Orbitale = 14 Qubits)
)

In [10]:
from rdkit import Chem
from rdkit.Chem import AllChem
import py3Dmol

# 3D-Struktur berechnen (Wasser erstellen und H-Atome sichtbar machen)
rkmol = Chem.AddHs(Chem.MolFromSmiles("O"))
AllChem.EmbedMolecule(rkmol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(rkmol)

# Koordinaten für PySCF berechnen
conf = rkmol.GetConformer()
atom_liste = []
for atom in rkmol.GetAtoms():
    pos = conf.GetAtomPosition(atom.GetIdx())
    atom_liste.append([atom.GetSymbol(), (pos.x, pos.y, pos.z)])
print(atom_liste)

# in PySCF laden
mol = pyscf.gto.Mole()
mol.build(atom=atom_liste, basis="sto-3g")

# 2. In MolBlock-Text umwandeln und im 3D-Viewer laden
mol_block = Chem.MolToMolBlock(rkmol)
viewer = py3Dmol.view(width=400, height=300)
viewer.addModel(mol_block, "mol")

# 3. Darstellungsstil wählen (Kugel-Stab-Modell)
viewer.setStyle({"stick": {}, "sphere": {"scale": 0.3}})
viewer.zoomTo()
viewer.show()


[['O', (0.007827036328440498, 0.39773799122489173, 0.0)], ['H', (-0.7672341344362831, -0.18384734761231294, 0.0)], ['H', (0.7594070981078427, -0.21389064361257837, 0.0)]]


NameError: name 'pyscf' is not defined

# Berechnen

In [ ]:
# Wir führen eine erste grobe klassische Berechnung durch (Hartree-Fock-Methode)
scf = pyscf.scf.RHF(mol).run()

# Wir lesen die wichtigsten physikalischen Werte aus dem Molekül aus:
num_orbitals = scf.mo_coeff.shape[1] # Anzahl der räumlichen Orbitale (hier: 4)
print(f"Anzahl der Orbitale: {num_orbitals}")

nelec = mol.nelec # Anzahl der Elektronen als Paar: (1 Alpha-Elektron, 1 Beta-Elektron)
print(f"Anzahl der Elektronen: {nelec}")

nuclear_repulsion_energy = mol.energy_nuc() # Abstoßungsenergie der beiden Atomkerne
print(f"Abstoßungsenergie: {nuclear_repulsion_energy}")


# Wir berechnen die Wechselwirkungs-Integrale der Elektronen (das "Spielfeld" der Kräfte)
hcore = scf.mo_coeff.T @ scf.get_hcore() @ scf.mo_coeff # Ein-Elektronen-Integrale (Bewegung + Kernanziehung)
eri = pyscf.ao2mo.kernel(mol, scf.mo_coeff) # Zwei-Elektronen-Integrale (gegenseitige Abstoßung)
eri = pyscf.ao2mo.restore(1, eri, num_orbitals) # Bringt die Abstoßungsdaten in ein 4D-Tabellenformat

print(f"Shape der Ein-Elektronen-Integrale: {hcore.shape}")
print(f"Shape der Zwei-Elektronen-Integrale: {eri.shape}")

# Wir machen eine klassische CCSD-Schätzung, um gute Start-Parameter für den Schaltkreis zu erhalten
ccsd = pyscf.cc.CCSD(scf).run()

converged SCF energy = -1.12675531719693
Anzahl der Orbitale: 4
Anzahl der Elektronen: (1, 1)
Abstoßungsenergie: 0.7151043390810812
Shape der Ein-Elektronen-Integrale: (4, 4)
Shape der Zwei-Elektronen-Integrale: (4, 4, 4, 4)
E(CCSD) = -1.151672678339738  E_corr = -0.02491736114280629
